# Fine-tune Whisper for Punjabi Speech Recognition

Fine-tunes OpenAI Whisper on real Punjabi speech from [aaparajit02/punjabi-asr](https://huggingface.co/datasets/aaparajit02/punjabi-asr) (the AI4Bharat "Shrutilipi" corpus - ~39K clips mined from All India Radio news broadcasts). Whisper's out-of-the-box Punjabi ASR accuracy is weaker than higher-resource languages; this follows the standard, widely-used Hugging Face "Fine-Tune Whisper for Multilingual ASR" recipe.

(Note: Mozilla Common Voice was originally used here, but as of October 2025 Mozilla discontinued distributing Common Voice through Hugging Face in favor of their own "Mozilla Data Collective" platform, so this notebook was switched to the dataset above instead.)

**Runtime > Change runtime type > select a GPU (T4 is fine on the free tier)** before running any cells below.

### Prerequisites
1. Create a (free) Hugging Face account.
2. Log in below (this dataset isn't gated, but logging in avoids anonymous rate limits on downloads).

In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer librosa soundfile huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import evaluate
import torch
from datasets import Audio, load_dataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

assert torch.cuda.is_available(), "No GPU detected - set Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

## Mount Google Drive

Checkpoints are saved here (not just Colab's local disk) so training progress survives a runtime disconnect/reset - only your Python variables are lost, not the saved files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Configuration

Use a smaller checkpoint (`small`/`medium`) for feasible fine-tuning on a single free-tier GPU. `large-v3` needs significantly more VRAM/time.

In [ ]:
BASE_MODEL = "openai/whisper-small"
LANGUAGE = "Punjabi"
TASK = "transcribe"
SAMPLING_RATE = 16000
EPOCHS = 3.0
BATCH_SIZE = 16
# Written to Drive (not local Colab disk) so checkpoints survive a runtime reset.
OUTPUT_DIR = f"/content/drive/MyDrive/whisper-finetune/{BASE_MODEL.split('/')[-1]}-punjabi"

# Set to a small number (e.g. 1000) for a quick end-to-end test run before
# committing to preprocessing/training on the full ~39K-clip dataset. None = use all.
NUM_TRAIN_EXAMPLES = 1000

## Load and preprocess Punjabi ASR data (Shrutilipi corpus)

In [ ]:
import os

print("Loading Punjabi ASR data (aaparajit02/punjabi-asr)...")
raw = load_dataset("aaparajit02/punjabi-asr", split="train")
if NUM_TRAIN_EXAMPLES is not None:
    raw = raw.select(range(min(NUM_TRAIN_EXAMPLES, len(raw))))

# This dataset ships a single 'train' split - carve out our own held-out test set.
split_raw = raw.train_test_split(test_size=0.02, seed=42)
common_voice = {"train": split_raw["train"], "test": split_raw["test"]}

# Keep only audio + transcript, resample to the 16kHz Whisper expects.
for split in common_voice:
    common_voice[split] = common_voice[split].select_columns(["audio", "transcript"])
    common_voice[split] = common_voice[split].cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language=LANGUAGE, task=TASK)

# Whisper's decoder caps labels at 448 tokens - truncate here as a hard guarantee
# training can never crash on an oversized label, regardless of dataset caching quirks.
MAX_LABEL_LENGTH = 448


def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(
        batch["transcript"], truncation=True, max_length=MAX_LABEL_LENGTH
    ).input_ids
    return batch


# Feature extraction is CPU-bound (the GPU sits idle for this step) - use all
# available cores instead of the previous num_proc=1, which serializes ~39K clips.
num_proc = max(1, os.cpu_count() - 1)
print(f"Preprocessing with num_proc={num_proc}...")
common_voice = {
    split: ds.map(prepare_dataset, remove_columns=ds.column_names, num_proc=num_proc)
    for split, ds in common_voice.items()
}

# Drop any example that got truncated above - keeps only examples whose full
# transcript fit, so we never train on an incomplete/mismatched transcript.
common_voice = {
    split: ds.filter(lambda x: len(x["labels"]) < MAX_LABEL_LENGTH)
    for split, ds in common_voice.items()
}

## Data collator and metric

In [ ]:
class DataCollatorSpeechSeq2SeqWithPadding:
    """Pads audio input features and label token sequences to the batch max,
    independently, since they have very different lengths/shapes."""

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)
wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## Load model and train

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = LANGUAGE.lower()
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=500,
    num_train_epochs=EPOCHS,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,  # newer transformers renamed tokenizer= to this
)

trainer.train()

## Save the fine-tuned model

Download the resulting folder (e.g. zip it) before your Colab session ends, since Colab storage isn't persistent.

In [ ]:
final_dir = OUTPUT_DIR + "-final"
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
print(f"Saved fine-tuned model to {final_dir}")